In [1]:
from pyspark.sql import functions as F
from datetime import datetime


# ============================================================
# AirOps 360 - Week 4 Task 27
# Silver end-to-end validation + evidence package
#
# PURPOSE:
# Independently validate the currently implemented Bronze ->
# Silver state before moving into Gold/BI implementation.
#
# THIS NOTEBOOK DOES NOT:
# - expand airport scope
# - expand month scope
# - build Gold
# - build Power BI
# ============================================================


spark.conf.set("spark.sql.session.timeZone", "UTC")


# ============================================================
# 0. CONFIGURATION / CONTRACT
# ============================================================

BATCH_KEY = "bts_reporting_carrier_ontime|2026|04"

EXPECTED_FLIGHT_ROWS = 597_919
EXPECTED_WEATHER_BRONZE_ROWS = 2
EXPECTED_WEATHER_ROWS = 1_440
EXPECTED_HOURS_PER_AIRPORT = 720

EXPECTED_PILOT_ORIGIN_FLIGHTS = 59_813
EXPECTED_OUTSIDE_PILOT_FLIGHTS = 538_106

WEATHER_AIRPORTS = ["ORD", "ATL"]

FLIGHT_KEY_VERSION = "flight_key_v1"
WEATHER_KEY_VERSION = "weather_key_v1"
ENRICHMENT_VERSION = "weather_enrichment_v1"


BRONZE_FLIGHTS_TABLE = (
    "lh_airops_bronze.brz_bts_flights"
)

BRONZE_WEATHER_TABLE = (
    "lh_airops_bronze.brz_weather_api_raw"
)

SILVER_STANDARDIZED_TABLE = (
    "slv_flights"
)

SILVER_VALIDATED_TABLE = (
    "slv_flights_validated"
)

SILVER_QUARANTINE_TABLE = (
    "slv_flights_quarantine"
)

FLIGHT_DQ_METRICS_TABLE = (
    "slv_flights_dq_metrics"
)

SILVER_WEATHER_TABLE = (
    "slv_weather_hourly"
)

SILVER_ENRICHED_TABLE = (
    "slv_flights_weather_enriched"
)

ENRICHMENT_METRICS_TABLE = (
    "slv_flights_weather_enrichment_metrics"
)

VALIDATION_METRICS_TABLE = (
    "slv_week4_validation_metrics"
)


print("=" * 80)
print("AIROPS 360 - TASK 27 SILVER END-TO-END VALIDATION")
print("=" * 80)

print("Flight Bronze    :", BRONZE_FLIGHTS_TABLE)
print("Silver std       :", SILVER_STANDARDIZED_TABLE)
print("Silver accepted  :", SILVER_VALIDATED_TABLE)
print("Silver quarantine:", SILVER_QUARANTINE_TABLE)
print("Weather Bronze   :", BRONZE_WEATHER_TABLE)
print("Weather Silver   :", SILVER_WEATHER_TABLE)
print("Enriched Silver  :", SILVER_ENRICHED_TABLE)


# ============================================================
# 1. READ CURRENT IMPLEMENTED STATE
# ============================================================

bronze_flights = (
    spark.table(
        BRONZE_FLIGHTS_TABLE
    )
    .filter(
        F.col("_bronze_batch_key")
        ==
        BATCH_KEY
    )
)


silver_standardized = (
    spark.table(
        SILVER_STANDARDIZED_TABLE
    )
    .filter(
        F.col("_bronze_batch_key")
        ==
        BATCH_KEY
    )
)


accepted = (
    spark.table(
        SILVER_VALIDATED_TABLE
    )
    .filter(
        F.col("_bronze_batch_key")
        ==
        BATCH_KEY
    )
)


quarantine = (
    spark.table(
        SILVER_QUARANTINE_TABLE
    )
)


flight_dq_metrics = (
    spark.table(
        FLIGHT_DQ_METRICS_TABLE
    )
    .filter(
        F.col("batch_key")
        ==
        BATCH_KEY
    )
    .orderBy(
        F.desc("measured_at_utc")
    )
    .limit(1)
)


bronze_weather = (
    spark.table(
        BRONZE_WEATHER_TABLE
    )
    .filter(
        F.col("airport_code").isin(
            WEATHER_AIRPORTS
        )
        &
        (
            F.col("request_start_date")
            ==
            F.lit("2026-04-01").cast("date")
        )
        &
        (
            F.col("request_end_date")
            ==
            F.lit("2026-04-30").cast("date")
        )
    )
)


weather = spark.table(
    SILVER_WEATHER_TABLE
)


enriched = spark.table(
    SILVER_ENRICHED_TABLE
)


enrichment_metrics = (
    spark.table(
        ENRICHMENT_METRICS_TABLE
    )
    .orderBy(
        F.desc("measured_at_utc")
    )
    .limit(1)
)


# ============================================================
# 2. BRONZE -> STANDARDIZED -> ACCEPTED RECONCILIATION
# ============================================================

bronze_rows = bronze_flights.count()

standardized_rows = (
    silver_standardized.count()
)

accepted_rows = accepted.count()

quarantine_rows = quarantine.count()

enriched_rows = enriched.count()


print("\nFLIGHT ROW RECONCILIATION")
print("-------------------------")

print(
    f"Bronze rows:       "
    f"{bronze_rows:,}"
)

print(
    f"Standardized rows: "
    f"{standardized_rows:,}"
)

print(
    f"Accepted rows:     "
    f"{accepted_rows:,}"
)

print(
    f"Quarantine rows:   "
    f"{quarantine_rows:,}"
)

print(
    f"Enriched rows:     "
    f"{enriched_rows:,}"
)


assert (
    bronze_rows
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    standardized_rows
    ==
    bronze_rows
), (
    "STOP: Bronze -> standardized "
    "row count changed"
)

assert (
    bronze_rows
    ==
    accepted_rows
    +
    quarantine_rows
), (
    "STOP: accepted + quarantine "
    "does not reconcile to Bronze"
)

assert (
    accepted_rows
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    quarantine_rows == 0
)

assert (
    enriched_rows
    ==
    accepted_rows
), (
    "STOP: weather enrichment changed "
    "accepted-flight grain"
)


print(
    "BRONZE -> SILVER RECONCILIATION: PASS"
)


# ============================================================
# 3. VERIFY PERSISTED FLIGHT DQ METRICS
#
# We do not merely trust the previous notebook output.
# We inspect the persisted metrics record as part of the
# end-to-end package.
# ============================================================

dq = (
    flight_dq_metrics
    .first()
    .asDict()
)


print("\nPERSISTED FLIGHT DQ METRICS")
print("---------------------------")

for k, v in dq.items():
    print(f"{k}: {v}")


assert dq["bronze_rows"] == EXPECTED_FLIGHT_ROWS

assert (
    dq["silver_input_rows"]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert dq["structural_invalid_rows"] == 0

assert (
    dq["duplicate_rows_quarantined"]
    ==
    0
)

assert dq["accepted_rows"] == EXPECTED_FLIGHT_ROWS

assert dq["quarantine_rows"] == 0

assert (
    dq["accepted_distinct_flight_keys"]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert dq["key_recompute_mismatches"] == 0

assert dq["reconciliation_pass"] is True

assert (
    dq["accepted_key_uniqueness_pass"]
    is True
)


print(
    "PERSISTED FLIGHT DQ METRICS: PASS"
)


# ============================================================
# 4. INDEPENDENT FLIGHT-KEY VALIDATION
#
# Recreate the EXACT Task-10 canonical key.
#
# flight_date
# + reporting_airline
# + flight_number
# + origin
# + dest
# + scheduled departure HHMM
# ============================================================

KEY_FIELDS = [
    "flight_date",
    "reporting_airline",
    "flight_number",
    "origin",
    "dest",
    "crs_dep_time_hhmm",
]


def flight_key_material():

    return F.concat(

        F.lit("flight_date="),

        F.date_format(
            "flight_date",
            "yyyy-MM-dd"
        ),

        F.lit(
            "|reporting_airline="
        ),

        F.upper(
            F.trim(
                F.col(
                    "reporting_airline"
                )
            )
        ),

        F.lit(
            "|flight_number="
        ),

        F.col(
            "flight_number"
        ).cast("string"),

        F.lit(
            "|origin="
        ),

        F.upper(
            F.trim(
                F.col("origin")
            )
        ),

        F.lit(
            "|dest="
        ),

        F.upper(
            F.trim(
                F.col("dest")
            )
        ),

        F.lit(
            "|crs_dep_time_hhmm="
        ),

        F.lpad(
            F.col(
                "crs_dep_time_hhmm"
            ).cast("string"),
            4,
            "0"
        )
    )


def flight_key_expr():

    return F.sha2(
        flight_key_material(),
        256
    )


accepted_null_keys = (
    accepted
    .filter(
        F.col("flight_key").isNull()
    )
    .count()
)


accepted_distinct_keys = (
    accepted
    .select("flight_key")
    .distinct()
    .count()
)


accepted_duplicate_groups = (
    accepted
    .groupBy("flight_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


distinct_business_keys = (
    accepted
    .select(*KEY_FIELDS)
    .distinct()
    .count()
)


flight_key_mismatches = (
    accepted
    .withColumn(
        "_validation_recomputed_key",
        flight_key_expr()
    )
    .filter(
        F.col("flight_key")
        !=
        F.col(
            "_validation_recomputed_key"
        )
    )
    .count()
)


print("\nFLIGHT KEY VALIDATION")
print("---------------------")

print(
    "Accepted NULL flight_key:      ",
    accepted_null_keys
)

print(
    "Distinct composite keys:       ",
    f"{distinct_business_keys:,}"
)

print(
    "Distinct flight_key values:    ",
    f"{accepted_distinct_keys:,}"
)

print(
    "Duplicate flight_key groups:   ",
    accepted_duplicate_groups
)

print(
    "Independent key mismatches:    ",
    flight_key_mismatches
)


assert accepted_null_keys == 0

assert (
    distinct_business_keys
    ==
    accepted_rows
)

assert (
    accepted_distinct_keys
    ==
    accepted_rows
)

assert accepted_duplicate_groups == 0

assert flight_key_mismatches == 0


flight_key_versions = {
    r[0]
    for r in (
        accepted
        .select(
            "_flight_key_version"
        )
        .distinct()
        .collect()
    )
}


assert flight_key_versions == {
    FLIGHT_KEY_VERSION
}


print(
    "DETERMINISTIC FLIGHT KEY: PASS"
)


# ============================================================
# 5. LINEAGE PRESERVATION
# ============================================================

FLIGHT_LINEAGE_COLS = [
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_source_file_name",
    "_bronze_source_hash",
    "_bronze_ingested_at_utc",
    "_bronze_load_year",
    "_bronze_load_month",
]


accepted_lineage_nulls = (
    accepted
    .agg(
        *[
            F.sum(
                F.col(c)
                .isNull()
                .cast("int")
            ).alias(c)
            for c in FLIGHT_LINEAGE_COLS
        ]
    )
    .first()
    .asDict()
)


accepted_lineage_null_total = sum(
    accepted_lineage_nulls.values()
)


print(
    "\nAccepted flight lineage NULLs:",
    accepted_lineage_nulls
)


assert accepted_lineage_null_total == 0


print(
    "FLIGHT LINEAGE PRESERVATION: PASS"
)


# ============================================================
# 6. BRONZE WEATHER -> SILVER WEATHER RECONCILIATION
# ============================================================

bronze_weather_rows = (
    bronze_weather.count()
)

weather_rows = weather.count()


weather_airport_counts = {
    r["airport_code"]: r["count"]
    for r in (
        weather
        .groupBy("airport_code")
        .count()
        .collect()
    )
}


print("\nWEATHER RECONCILIATION")
print("----------------------")

print(
    "Bronze response rows:",
    bronze_weather_rows
)

print(
    "Silver hourly rows:",
    weather_rows
)

print(
    "Hourly rows by airport:",
    weather_airport_counts
)


assert (
    bronze_weather_rows
    ==
    EXPECTED_WEATHER_BRONZE_ROWS
)

assert (
    weather_rows
    ==
    EXPECTED_WEATHER_ROWS
)

assert weather_airport_counts == {
    "ATL": EXPECTED_HOURS_PER_AIRPORT,
    "ORD": EXPECTED_HOURS_PER_AIRPORT,
}


print(
    "BRONZE -> SILVER WEATHER: PASS"
)


# ============================================================
# 7. INDEPENDENT WEATHER-KEY + UNIQUENESS VALIDATION
# ============================================================

weather_null_keys = (
    weather
    .filter(
        F.col("weather_key").isNull()
    )
    .count()
)


weather_distinct_keys = (
    weather
    .select("weather_key")
    .distinct()
    .count()
)


weather_duplicate_local_groups = (
    weather
    .groupBy(
        "airport_code",
        "weather_hour_local",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


weather_duplicate_utc_groups = (
    weather
    .groupBy(
        "airport_code",
        "weather_hour_utc",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


weather_key_mismatches = (
    weather
    .withColumn(
        "_validation_recomputed_weather_key",

        F.sha2(
            F.concat_ws(
                "|",

                F.upper(
                    F.trim(
                        F.col(
                            "airport_code"
                        )
                    )
                ),

                F.date_format(
                    F.col(
                        "weather_hour_local"
                    ),
                    "yyyy-MM-dd HH:mm:ss",
                ),
            ),
            256,
        )
    )
    .filter(
        F.col("weather_key")
        !=
        F.col(
            "_validation_recomputed_weather_key"
        )
    )
    .count()
)


weather_key_versions = {
    r[0]
    for r in (
        weather
        .select(
            "_weather_key_version"
        )
        .distinct()
        .collect()
    )
}


print("\nWEATHER KEY / GRAIN VALIDATION")
print("------------------------------")

print(
    "Weather NULL keys:             ",
    weather_null_keys
)

print(
    "Distinct weather_key:          ",
    weather_distinct_keys
)

print(
    "Duplicate local airport-hours: ",
    weather_duplicate_local_groups
)

print(
    "Duplicate UTC airport-hours:   ",
    weather_duplicate_utc_groups
)

print(
    "Weather-key mismatches:        ",
    weather_key_mismatches
)


assert weather_null_keys == 0

assert (
    weather_distinct_keys
    ==
    weather_rows
)

assert (
    weather_duplicate_local_groups
    ==
    0
)

assert (
    weather_duplicate_utc_groups
    ==
    0
)

assert weather_key_mismatches == 0

assert weather_key_versions == {
    WEATHER_KEY_VERSION
}


print(
    "DETERMINISTIC WEATHER GRAIN: PASS"
)


# ============================================================
# 8. WEATHER VALUE / LINEAGE SANITY
# ============================================================

WEATHER_VALUE_COLS = [
    "temperature_2m_c",
    "relative_humidity_2m_pct",
    "precipitation_mm",
    "snowfall_cm",
    "weather_code",
    "cloud_cover_pct",
    "wind_speed_10m_kmh",
    "wind_direction_10m_deg",
]


weather_value_null_counts = (
    weather
    .agg(
        *[
            F.sum(
                F.col(c)
                .isNull()
                .cast("int")
            ).alias(c)
            for c in WEATHER_VALUE_COLS
        ]
    )
    .first()
    .asDict()
)


weather_value_null_total = sum(
    weather_value_null_counts.values()
)


print(
    "\nWeather-value NULL counts:",
    weather_value_null_counts
)


assert weather_value_null_total == 0


print(
    "WEATHER VALUE QA: PASS"
)


# ============================================================
# 9. ENRICHMENT CARDINALITY VALIDATION
# ============================================================

enriched_distinct_keys = (
    enriched
    .select("flight_key")
    .distinct()
    .count()
)


enriched_duplicate_flight_groups = (
    enriched
    .groupBy("flight_key")
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)


wrong_airport_matches = (
    enriched
    .filter(
        F.col(
            "origin_weather_key"
        ).isNotNull()
        &
        (
            F.upper(
                F.trim(
                    F.col("origin")
                )
            )
            !=
            F.col(
                "wx_airport_code"
            )
        )
    )
    .count()
)


wrong_hour_matches = (
    enriched
    .filter(
        F.col(
            "origin_weather_key"
        ).isNotNull()
        &
        (
            F.col(
                "origin_sched_dep_hour_local"
            )
            !=
            F.col(
                "origin_weather_hour_local"
            )
        )
    )
    .count()
)


matched_rows = (
    enriched
    .filter(
        F.col(
            "origin_weather_match_status"
        )
        ==
        "MATCHED"
    )
    .count()
)


pilot_unmatched_rows = (
    enriched
    .filter(
        F.col(
            "origin_weather_match_status"
        )
        ==
        "PILOT_AIRPORT_HOUR_UNMATCHED"
    )
    .count()
)


outside_pilot_rows = (
    enriched
    .filter(
        F.col(
            "origin_weather_match_status"
        )
        ==
        "OUTSIDE_WEATHER_PILOT_AIRPORT"
    )
    .count()
)


print("\nENRICHMENT CARDINALITY")
print("----------------------")

print(
    "Accepted rows before enrichment:",
    f"{accepted_rows:,}"
)

print(
    "Enriched rows:                 ",
    f"{enriched_rows:,}"
)

print(
    "Distinct enriched flight_key:  ",
    f"{enriched_distinct_keys:,}"
)

print(
    "Duplicate flight groups:       ",
    enriched_duplicate_flight_groups
)

print(
    "Weather matched:               ",
    f"{matched_rows:,}"
)

print(
    "Pilot-airport unmatched:       ",
    f"{pilot_unmatched_rows:,}"
)

print(
    "Outside-pilot preserved:       ",
    f"{outside_pilot_rows:,}"
)

print(
    "Wrong-airport matches:         ",
    wrong_airport_matches
)

print(
    "Wrong-hour matches:            ",
    wrong_hour_matches
)


assert enriched_rows == accepted_rows

assert (
    enriched_distinct_keys
    ==
    accepted_rows
)

assert enriched_duplicate_flight_groups == 0

assert wrong_airport_matches == 0

assert wrong_hour_matches == 0

assert (
    matched_rows
    ==
    EXPECTED_PILOT_ORIGIN_FLIGHTS
)

assert pilot_unmatched_rows == 0

assert (
    outside_pilot_rows
    ==
    EXPECTED_OUTSIDE_PILOT_FLIGHTS
)

assert (
    matched_rows
    +
    pilot_unmatched_rows
    +
    outside_pilot_rows
    ==
    enriched_rows
)


print(
    "ENRICHMENT CARDINALITY: PASS"
)


# ============================================================
# 10. VERIFY PERSISTED ENRICHMENT METRICS
# ============================================================

em = (
    enrichment_metrics
    .first()
    .asDict()
)


print("\nPERSISTED ENRICHMENT METRICS")
print("----------------------------")

for k, v in em.items():
    print(f"{k}: {v}")


assert (
    em["flight_rows_before"]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    em["flight_rows_after"]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    em[
        "distinct_flight_keys_before"
    ]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    em[
        "distinct_flight_keys_after"
    ]
    ==
    EXPECTED_FLIGHT_ROWS
)

assert (
    em[
        "weather_duplicate_airport_hour_groups"
    ]
    ==
    0
)

assert (
    em["duplicate_flight_groups_after"]
    ==
    0
)

assert em["wrong_airport_matches"] == 0

assert em["wrong_hour_matches"] == 0

assert em["row_count_preserved"] is True

assert (
    em["flight_key_grain_preserved"]
    is True
)


print(
    "PERSISTED ENRICHMENT METRICS: PASS"
)


# ============================================================
# 11. CREATE A CONCISE VALIDATION EVIDENCE TABLE
#
# This becomes a small machine-readable Week 4 evidence
# package inside Silver itself.
# ============================================================

validation_rows = [

    (
        "bronze_flight_rows",
        str(bronze_rows),
        str(EXPECTED_FLIGHT_ROWS),
        "PASS",
    ),

    (
        "standardized_flight_rows",
        str(standardized_rows),
        str(EXPECTED_FLIGHT_ROWS),
        "PASS",
    ),

    (
        "accepted_flight_rows",
        str(accepted_rows),
        str(EXPECTED_FLIGHT_ROWS),
        "PASS",
    ),

    (
        "quarantine_rows",
        str(quarantine_rows),
        "0",
        "PASS",
    ),

    (
        "flight_key_distinct",
        str(accepted_distinct_keys),
        str(EXPECTED_FLIGHT_ROWS),
        "PASS",
    ),

    (
        "flight_key_recompute_mismatches",
        str(flight_key_mismatches),
        "0",
        "PASS",
    ),

    (
        "bronze_weather_responses",
        str(bronze_weather_rows),
        "2",
        "PASS",
    ),

    (
        "silver_weather_rows",
        str(weather_rows),
        "1440",
        "PASS",
    ),

    (
        "weather_key_distinct",
        str(weather_distinct_keys),
        "1440",
        "PASS",
    ),

    (
        "weather_local_duplicate_groups",
        str(
            weather_duplicate_local_groups
        ),
        "0",
        "PASS",
    ),

    (
        "enriched_flight_rows",
        str(enriched_rows),
        str(EXPECTED_FLIGHT_ROWS),
        "PASS",
    ),

    (
        "enriched_duplicate_flight_groups",
        str(
            enriched_duplicate_flight_groups
        ),
        "0",
        "PASS",
    ),

    (
        "pilot_weather_matches",
        str(matched_rows),
        "59813",
        "PASS",
    ),

    (
        "outside_pilot_flights_preserved",
        str(outside_pilot_rows),
        "538106",
        "PASS",
    ),

    (
        "wrong_airport_matches",
        str(wrong_airport_matches),
        "0",
        "PASS",
    ),

    (
        "wrong_hour_matches",
        str(wrong_hour_matches),
        "0",
        "PASS",
    ),
]


validation_evidence = (
    spark.createDataFrame(
        validation_rows,
        [
            "check_name",
            "observed_value",
            "expected_value",
            "status",
        ]
    )
    .withColumn(
        "validated_at_utc",
        F.current_timestamp()
    )
    .withColumn(
        "validation_version",
        F.lit(
            "week4_silver_validation_v1"
        )
    )
)


(
    validation_evidence.write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        VALIDATION_METRICS_TABLE
    )
)


# ============================================================
# 12. POST-WRITE EVIDENCE-TABLE QA
# ============================================================

validation_target = spark.table(
    VALIDATION_METRICS_TABLE
)


validation_check_count = (
    validation_target.count()
)


failed_validation_checks = (
    validation_target
    .filter(
        F.col("status") != "PASS"
    )
    .count()
)


assert validation_check_count == len(
    validation_rows
)

assert failed_validation_checks == 0


print(
    "\nWEEK 4 VALIDATION EVIDENCE TABLE: PASS"
)

print(
    "Evidence table:",
    VALIDATION_METRICS_TABLE
)


# ============================================================
# 13. FINAL END-TO-END EVIDENCE
# ============================================================

print("\n")
print("=" * 84)

print(
    "AIROPS 360 - WEEK 4 SILVER "
    "END-TO-END VALIDATION"
)

print("=" * 84)

print("\nFLIGHTS")

print(
    f"Bronze flight rows:              "
    f"{bronze_rows:,}"
)

print(
    f"Standardized Silver rows:        "
    f"{standardized_rows:,}"
)

print(
    f"Accepted Silver rows:            "
    f"{accepted_rows:,}"
)

print(
    f"Quarantine rows:                 "
    f"{quarantine_rows:,}"
)

print(
    f"Distinct flight_key:             "
    f"{accepted_distinct_keys:,}"
)

print(
    f"Flight-key recompute mismatches: "
    f"{flight_key_mismatches:,}"
)


print("\nWEATHER")

print(
    f"Bronze weather response rows:    "
    f"{bronze_weather_rows:,}"
)

print(
    f"Silver hourly weather rows:      "
    f"{weather_rows:,}"
)

print(
    f"ATL hourly rows:                 "
    f"{weather_airport_counts['ATL']:,}"
)

print(
    f"ORD hourly rows:                 "
    f"{weather_airport_counts['ORD']:,}"
)

print(
    f"Distinct weather_key:            "
    f"{weather_distinct_keys:,}"
)

print(
    f"Duplicate local airport-hours:   "
    f"{weather_duplicate_local_groups:,}"
)

print(
    f"Duplicate UTC airport-hours:     "
    f"{weather_duplicate_utc_groups:,}"
)

print(
    f"Weather-key mismatches:          "
    f"{weather_key_mismatches:,}"
)


print("\nENRICHMENT")

print(
    f"Flight rows before join:         "
    f"{accepted_rows:,}"
)

print(
    f"Flight rows after join:          "
    f"{enriched_rows:,}"
)

print(
    f"Distinct flight_key after join:  "
    f"{enriched_distinct_keys:,}"
)

print(
    f"Duplicate flight groups:         "
    f"{enriched_duplicate_flight_groups:,}"
)

print(
    f"ORD/ATL weather matches:         "
    f"{matched_rows:,}"
)

print(
    f"ORD/ATL unmatched:               "
    f"{pilot_unmatched_rows:,}"
)

print(
    f"Outside-pilot flights preserved: "
    f"{outside_pilot_rows:,}"
)

print(
    f"Wrong-airport matches:           "
    f"{wrong_airport_matches:,}"
)

print(
    f"Wrong-hour matches:              "
    f"{wrong_hour_matches:,}"
)


print("\nSCOPE BOUNDARY")

print(
    "Gold physical tables:           "
    "NOT VALIDATED / NOT CLAIMED"
)

print(
    "Direct Lake / Power BI:         "
    "NOT VALIDATED / NOT CLAIMED"
)


print("=" * 84)

print(
    "\nTASK 27 STATUS: PASS"
)


# ============================================================
# 14. HUMAN-READABLE EVIDENCE PACKAGE
# ============================================================

print(
    "\nWeek 4 validation checks:"
)

display(
    validation_target
    .orderBy(
        "check_name"
    )
)


print(
    "\nFlight DQ persisted metrics:"
)

display(
    flight_dq_metrics
)


print(
    "\nWeather-enrichment persisted metrics:"
)

display(
    enrichment_metrics
)

StatementMeta(, fa6ed9ab-e3c6-489a-9cae-44bde93dfa93, 3, Finished, Available, Finished, True)

AIROPS 360 - TASK 27 SILVER END-TO-END VALIDATION
Flight Bronze    : lh_airops_bronze.brz_bts_flights
Silver std       : slv_flights
Silver accepted  : slv_flights_validated
Silver quarantine: slv_flights_quarantine
Weather Bronze   : lh_airops_bronze.brz_weather_api_raw
Weather Silver   : slv_weather_hourly
Enriched Silver  : slv_flights_weather_enriched

FLIGHT ROW RECONCILIATION
-------------------------
Bronze rows:       597,919
Standardized rows: 597,919
Accepted rows:     597,919
Quarantine rows:   0
Enriched rows:     597,919
BRONZE -> SILVER RECONCILIATION: PASS

PERSISTED FLIGHT DQ METRICS
---------------------------
measured_at_utc: 2026-09-17 04:45:34.167614
batch_key: bts_reporting_carrier_ontime|2026|04
flight_key_version: flight_key_v1
dq_ruleset_version: dq_rules_v1
bronze_rows: 597919
silver_input_rows: 597919
structural_invalid_rows: 0
duplicate_rows_quarantined: 0
accepted_rows: 597919
quarantine_rows: 0
accepted_distinct_flight_keys: 597919
key_recompute_mismatches:

SynapseWidget(Synapse.DataFrame, 97b9c59c-f7ff-43ed-944e-ee6d95f087c9)


Flight DQ persisted metrics:


SynapseWidget(Synapse.DataFrame, 37d40cb5-99fe-4abb-a66b-e6d01f3eed7d)


Weather-enrichment persisted metrics:


SynapseWidget(Synapse.DataFrame, 2a0417da-7c9a-408c-b68a-cf5cdf9e8837)